In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.svm import SVC
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.metrics import Precision, Recall


def load_product_data(product):
    """
    product: 'cn7' 또는 'rg3'
    PDF의 DataLoader에 해당하는 부분.
    """
    labeled = pd.read_csv(f"moldset_labeled_{product}.csv")
    unlabeled = pd.read_csv(f"moldset_unlabeled_{product}.csv")

    # PDF에서 CSV 저장 후 다시 불러올 때 생기는 인덱스 열 제거
    labeled = labeled.loc[
        :, ~labeled.columns.str.startswith("Unnamed:")
    ]
    unlabeled = unlabeled.loc[
        :, ~unlabeled.columns.str.startswith("Unnamed:")
    ]

    if "PassOrFail" not in labeled.columns:
        raise ValueError(f"{product}: 라벨 파일에 PassOrFail 열이 없습니다.")

    X = labeled.drop(columns="PassOrFail")
    y = labeled["PassOrFail"].astype(int)

    missing = X.columns.difference(unlabeled.columns)
    if len(missing):
        raise ValueError(
            f"{product}: 미라벨 파일에 없는 입력 열: {list(missing)}"
        )

    # 학습 데이터와 입력 열 순서를 동일하게 맞춤
    unlabeled = unlabeled[X.columns].copy()

    # PDF: 라벨 데이터는 양품/불량 비율을 유지하며 70:30 분리
    splitter = StratifiedShuffleSplit(
        n_splits=1,
        test_size=0.3,
        random_state=42,
    )

    train_idx, test_idx = next(splitter.split(X, y))

    X_train = X.iloc[train_idx].reset_index(drop=True)
    y_train = y.iloc[train_idx].reset_index(drop=True)
    X_test = X.iloc[test_idx].reset_index(drop=True)
    y_test = y.iloc[test_idx].reset_index(drop=True)

    print(
        f"{product.upper()}: 라벨 학습 {len(X_train)}건, "
        f"평가 {len(X_test)}건, 미라벨 {len(unlabeled)}건"
    )
    return X_train, y_train, X_test, y_test, unlabeled


def evaluate_model(y_true, y_pred):
    """PDF의 evaluation 함수에 해당."""
    print("혼동행렬:\n", confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ))
    print("정확도:", accuracy_score(y_true, y_pred))
    print("정밀도:", precision_score(
        y_true, y_pred, zero_division=0
    ))
    print("재현율:", recall_score(
        y_true, y_pred, zero_division=0
    ))
    print("F1:", f1_score(
        y_true, y_pred, zero_division=0
    ))

    if len(np.unique(y_true)) == 2:
        print("ROC-AUC:", roc_auc_score(y_true, y_pred))

I0000 00:00:1790572435.304600     124 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790572457.873538     124 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
def train_svc_semi_supervised(
    product,
    percentage=10,
    unlabeled_usage=90,
):
    X_train, y_train, X_test, y_test, unlabeled = (
        load_product_data(product)
    )

    X_with_label = X_train.copy()
    y_with_label = y_train.copy()
    without_label = unlabeled.copy()

    # 전체 미라벨 데이터 중 90%를 사용할 경우 10%가 남으면 종료
    target_remaining = int(
        len(unlabeled) * (100 - unlabeled_usage) / 100
    )

    round_number = 0

    while len(without_label) > target_remaining:
        round_number += 1

        model = SVC(
            C=0.001,
            kernel="rbf",
            gamma=1e-2,
            class_weight={0: 100.0, 1: 1.0},
            probability=True,
            random_state=42,
        )

        model.fit(X_with_label, y_with_label)

        # PDF의 confident_prediction(): 가장 큰 클래스 확률
        probabilities = model.predict_proba(without_label)
        confidence = probabilities.max(axis=1)

        ranked = without_label.copy()
        ranked["confidence"] = confidence
        ranked = ranked.sort_values(
            by="confidence",
            ascending=False,
        )

        count = max(
            1,
            int(len(ranked) * percentage / 100),
        )
        count = min(
            count,
            len(ranked) - target_remaining,
        )

        chosen = ranked.iloc[:count].drop(columns="confidence")
        without_label = ranked.iloc[count:].drop(
            columns="confidence"
        ).copy()

        # 모델의 예측값을 임시 라벨로 부여
        pseudo_labels = model.predict(chosen)

        X_with_label = pd.concat(
            [X_with_label, chosen],
            ignore_index=True,
        )
        y_with_label = pd.concat(
            [y_with_label, pd.Series(pseudo_labels)],
            ignore_index=True,
        )

        print(
            f"[{product.upper()} SVC] {round_number}회차: "
            f"임시 라벨 {count}건 추가, "
            f"미라벨 {len(without_label)}건 남음"
        )

    # 마지막 회차에 추가된 임시 라벨도 반영
    model.fit(X_with_label, y_with_label)

    print(f"\n[{product.upper()} SVC] 최종 평가")
    evaluate_model(y_test, model.predict(X_test))
    return model

## Deep Learning 기반 모델

In [3]:
def make_keras_model(input_dim):
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(32, activation="relu"),
        Dense(64, activation="relu"),
        Dropout(0.25),
        Dense(32, activation="relu"),
        Dropout(0.2),
        Dense(16, activation="relu"),
        Dense(1, activation="sigmoid"),
    ])

    model.compile(
        loss="binary_crossentropy",
        optimizer="adam",
        metrics=[
            "accuracy",
            Precision(name="precision"),
            Recall(name="recall"),
        ],
    )
    return model


def fit_keras_like_guide(
    model,
    X,
    y,
    max_epochs=100,
    patience=10,
):
    """
    PDF처럼 매번 1 epoch씩 학습하고
    val_precision과 val_recall을 함께 관찰.
    """
    best_precision = 0.0
    best_recall = 0.0
    no_improvement = 0

    for epoch in range(max_epochs):
        history = model.fit(
            X,
            y,
            epochs=1,
            batch_size=32,
            validation_split=0.3,
            verbose=0,
        )

        val_precision = history.history["val_precision"][0]
        val_recall = history.history["val_recall"][0]

        if (
            val_precision >= best_precision
            and val_recall >= best_recall
        ):
            best_precision = val_precision
            best_recall = val_recall
            no_improvement = 0
        else:
            no_improvement += 1

        if no_improvement >= patience:
            break

    print(
        f"학습 {epoch + 1} epoch, "
        f"검증 정밀도 {val_precision:.3f}, "
        f"검증 재현율 {val_recall:.3f}"
    )


def train_keras_semi_supervised(
    product,
    percentage=10,
    unlabeled_usage=90,
):
    tf.keras.utils.set_random_seed(42)

    X_train, y_train, X_test, y_test, unlabeled = (
        load_product_data(product)
    )

    X_with_label = X_train.copy()
    y_with_label = y_train.copy()
    without_label = unlabeled.copy()

    target_remaining = int(
        len(unlabeled) * (100 - unlabeled_usage) / 100
    )

    model = make_keras_model(X_train.shape[1])
    round_number = 0

    while len(without_label) > target_remaining:
        round_number += 1

        # PDF: 현재 라벨 데이터를 이용해 신경망 학습
        fit_keras_like_guide(
            model,
            X_with_label,
            y_with_label,
        )

        # 최신 Keras에서는 PDF의 predict_proba() 대신 predict() 사용
        probability_of_defect = model.predict(
            without_label,
            verbose=0,
        ).ravel()

        confidence = np.maximum(
            probability_of_defect,
            1 - probability_of_defect,
        )

        ranked = without_label.copy()
        ranked["confidence"] = confidence
        ranked = ranked.sort_values(
            by="confidence",
            ascending=False,
        )

        count = max(
            1,
            int(len(ranked) * percentage / 100),
        )
        count = min(
            count,
            len(ranked) - target_remaining,
        )

        chosen = ranked.iloc[:count].drop(columns="confidence")
        without_label = ranked.iloc[count:].drop(
            columns="confidence"
        ).copy()

        chosen_probabilities = model.predict(
            chosen,
            verbose=0,
        ).ravel()
        pseudo_labels = (
            chosen_probabilities >= 0.5
        ).astype(int)

        X_with_label = pd.concat(
            [X_with_label, chosen],
            ignore_index=True,
        )
        y_with_label = pd.concat(
            [y_with_label, pd.Series(pseudo_labels)],
            ignore_index=True,
        )

        print(
            f"[{product.upper()} Keras] {round_number}회차: "
            f"임시 라벨 {count}건 추가, "
            f"미라벨 {len(without_label)}건 남음"
        )

    # 마지막으로 추가한 임시 라벨까지 학습에 반영
    fit_keras_like_guide(model, X_with_label, y_with_label)

    test_probabilities = model.predict(
        X_test,
        verbose=0,
    ).ravel()
    test_predictions = (
        test_probabilities >= 0.5
    ).astype(int)

    print(f"\n[{product.upper()} Keras] 최종 평가")
    evaluate_model(y_test, test_predictions)
    return model

In [4]:
svc_cn7 = train_svc_semi_supervised("cn7")
keras_cn7 = train_keras_semi_supervised("cn7")

CN7: 라벨 학습 847건, 평가 364건, 미라벨 35239건
[CN7 SVC] 1회차: 임시 라벨 3523건 추가, 미라벨 31716건 남음
[CN7 SVC] 2회차: 임시 라벨 3171건 추가, 미라벨 28545건 남음
[CN7 SVC] 3회차: 임시 라벨 2854건 추가, 미라벨 25691건 남음
[CN7 SVC] 4회차: 임시 라벨 2569건 추가, 미라벨 23122건 남음
[CN7 SVC] 5회차: 임시 라벨 2312건 추가, 미라벨 20810건 남음
[CN7 SVC] 6회차: 임시 라벨 2081건 추가, 미라벨 18729건 남음
[CN7 SVC] 7회차: 임시 라벨 1872건 추가, 미라벨 16857건 남음
[CN7 SVC] 8회차: 임시 라벨 1685건 추가, 미라벨 15172건 남음
[CN7 SVC] 9회차: 임시 라벨 1517건 추가, 미라벨 13655건 남음
[CN7 SVC] 10회차: 임시 라벨 1365건 추가, 미라벨 12290건 남음
[CN7 SVC] 11회차: 임시 라벨 1229건 추가, 미라벨 11061건 남음
[CN7 SVC] 12회차: 임시 라벨 1106건 추가, 미라벨 9955건 남음
[CN7 SVC] 13회차: 임시 라벨 995건 추가, 미라벨 8960건 남음
[CN7 SVC] 14회차: 임시 라벨 896건 추가, 미라벨 8064건 남음
[CN7 SVC] 15회차: 임시 라벨 806건 추가, 미라벨 7258건 남음
[CN7 SVC] 16회차: 임시 라벨 725건 추가, 미라벨 6533건 남음
[CN7 SVC] 17회차: 임시 라벨 653건 추가, 미라벨 5880건 남음
[CN7 SVC] 18회차: 임시 라벨 588건 추가, 미라벨 5292건 남음
[CN7 SVC] 19회차: 임시 라벨 529건 추가, 미라벨 4763건 남음
[CN7 SVC] 20회차: 임시 라벨 476건 추가, 미라벨 4287건 남음
[CN7 SVC] 21회차: 임시 라벨 428건 추가, 미라벨 3859건 남음
[CN7 SVC] 22회차: 임시 라벨 336

## 로지스틱 회귀

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

def run_logistic_semi(product="cn7"):
    X_train, y_train, X_test, y_test, X_unlabeled = load_product_data(product)

    # 라벨 학습 데이터로만 스케일러 학습
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    X_unlabeled = scaler.transform(X_unlabeled)

    y_train = y_train.to_numpy()
    y_test = y_test.to_numpy()

    # ① 라벨 데이터만 학습
    base = LogisticRegression(max_iter=1000, class_weight="balanced")
    base.fit(X_train, y_train)

    print(f"\n[{product.upper()} 로지스틱 회귀: 라벨만]")
    base_prob = base.predict_proba(X_test)[:, 1]
    evaluate_model(y_test, (base_prob >= 0.5).astype(int))
    print("확률 기반 ROC-AUC:", roc_auc_score(y_test, base_prob))

    # ② 확신도 0.95 이상인 미라벨 데이터 중 최대 10% 선택
    unlabeled_prob = base.predict_proba(X_unlabeled)[:, 1]
    confidence = np.maximum(unlabeled_prob, 1 - unlabeled_prob)
    candidates = np.flatnonzero(confidence >= 0.95)
    max_count = max(1, int(len(X_unlabeled) * 0.10))
    selected = candidates[np.argsort(confidence[candidates])[::-1][:max_count]]

    if len(selected) == 0:
        print("선택된 임시 라벨이 없어 준지도 재학습을 건너뜁니다.")
        return base

    X_pseudo = X_unlabeled[selected]
    y_pseudo = (unlabeled_prob[selected] >= 0.5).astype(int)
    print("임시 라벨 개수:", pd.Series(y_pseudo).value_counts().to_dict())

    # 실제 라벨 가중치 1, 임시 라벨 가중치 0.2
    X_combined = np.concatenate([X_train, X_pseudo])
    y_combined = np.concatenate([y_train, y_pseudo])
    sample_weights = np.concatenate([
        np.ones(len(y_train)),
        np.full(len(y_pseudo), 0.2),
    ])

    # ③ 임시 라벨을 더해 새 모델 학습
    semi = LogisticRegression(max_iter=1000, class_weight="balanced")
    semi.fit(X_combined, y_combined, sample_weight=sample_weights)

    print(f"\n[{product.upper()} 로지스틱 회귀: 준지도]")
    semi_prob = semi.predict_proba(X_test)[:, 1]
    evaluate_model(y_test, (semi_prob >= 0.5).astype(int))
    print("확률 기반 ROC-AUC:", roc_auc_score(y_test, semi_prob))
    return semi

logistic_cn7 = run_logistic_semi("cn7")
# logistic_rg3 = run_logistic_semi("rg3")

CN7: 라벨 학습 847건, 평가 364건, 미라벨 35239건

[CN7 로지스틱 회귀: 라벨만]
혼동행렬:
 [[347  12]
 [  2   3]]
정확도: 0.9615384615384616
정밀도: 0.2
재현율: 0.6
F1: 0.3
ROC-AUC: 0.7832869080779945
확률 기반 ROC-AUC: 0.9615598885793871
임시 라벨 개수: {0: 3523}

[CN7 로지스틱 회귀: 준지도]
혼동행렬:
 [[342  17]
 [  1   4]]
정확도: 0.9505494505494505
정밀도: 0.19047619047619047
재현율: 0.8
F1: 0.3076923076923077
ROC-AUC: 0.8763231197771588
확률 기반 ROC-AUC: 0.9554317548746518


In [10]:
## 랜덤 포레스트

In [12]:
from sklearn.ensemble import RandomForestClassifier

def run_forest_semi(product="cn7"):
    X_train, y_train, X_test, y_test, X_unlabeled = load_product_data(product)

    y_train = y_train.to_numpy()
    y_test = y_test.to_numpy()

    def make_forest():
        return RandomForestClassifier(
            n_estimators=200,
            max_depth=6,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        )

    # ① 라벨 데이터만 학습
    base = make_forest()
    base.fit(X_train, y_train)

    print(f"\n[{product.upper()} 랜덤 포레스트: 라벨만]")
    base_prob = base.predict_proba(X_test)[:, 1]
    evaluate_model(y_test, (base_prob >= 0.5).astype(int))
    print("확률 기반 ROC-AUC:", roc_auc_score(y_test, base_prob))

    # ② 확신도 0.95 이상인 미라벨 데이터 중 최대 10% 선택
    unlabeled_prob = base.predict_proba(X_unlabeled)[:, 1]
    confidence = np.maximum(unlabeled_prob, 1 - unlabeled_prob)
    candidates = np.flatnonzero(confidence >= 0.95)
    max_count = max(1, int(len(X_unlabeled) * 0.10))
    selected = candidates[np.argsort(confidence[candidates])[::-1][:max_count]]

    if len(selected) == 0:
        print("선택된 임시 라벨이 없어 준지도 재학습을 건너뜁니다.")
        return base

    X_pseudo = X_unlabeled.iloc[selected]
    y_pseudo = (unlabeled_prob[selected] >= 0.5).astype(int)
    print("임시 라벨 개수:", pd.Series(y_pseudo).value_counts().to_dict())

    # ③ 임시 라벨을 더해 새 모델 학습
    X_combined = pd.concat(
        [X_train, X_pseudo], ignore_index=True
    )
    y_combined = np.concatenate([y_train, y_pseudo])
    sample_weights = np.concatenate([
        np.ones(len(y_train)),
        np.full(len(y_pseudo), 0.2),
    ])

    semi = make_forest()
    semi.fit(X_combined, y_combined, sample_weight=sample_weights)

    print(f"\n[{product.upper()} 랜덤 포레스트: 준지도]")
    semi_prob = semi.predict_proba(X_test)[:, 1]
    evaluate_model(y_test, (semi_prob >= 0.5).astype(int))
    print("확률 기반 ROC-AUC:", roc_auc_score(y_test, semi_prob))
    return semi

forest_cn7 = run_forest_semi("cn7")
# forest_rg3 = run_forest_semi("rg3")

CN7: 라벨 학습 847건, 평가 364건, 미라벨 35239건

[CN7 랜덤 포레스트: 라벨만]
혼동행렬:
 [[356   3]
 [  3   2]]
정확도: 0.9835164835164835
정밀도: 0.4
재현율: 0.4
F1: 0.4
ROC-AUC: 0.6958217270194986
확률 기반 ROC-AUC: 0.9805013927576602
임시 라벨 개수: {0: 3523}

[CN7 랜덤 포레스트: 준지도]
혼동행렬:
 [[353   6]
 [  3   2]]
정확도: 0.9752747252747253
정밀도: 0.25
재현율: 0.4
F1: 0.3076923076923077
ROC-AUC: 0.6916434540389972
확률 기반 ROC-AUC: 0.9576601671309192


In [13]:
## 경량화 모델

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import EarlyStopping

def run_small_keras_semi(product="cn7"):
    X_train, y_train, X_test, y_test, X_unlabeled = load_product_data(product)

    y_train = y_train.to_numpy()
    y_test = y_test.to_numpy()

    # Keras의 조기 종료용 검증 데이터는 라벨 학습 데이터에서 분리
    X_fit, X_val, y_fit, y_val = train_test_split(
        X_train,
        y_train,
        test_size=0.25,
        stratify=y_train,
        random_state=42,
    )

    scaler = StandardScaler()
    X_fit = scaler.fit_transform(X_fit).astype("float32")
    X_val = scaler.transform(X_val).astype("float32")
    X_test = scaler.transform(X_test).astype("float32")
    X_unlabeled = scaler.transform(X_unlabeled).astype("float32")

    weights = compute_class_weight(
        class_weight="balanced",
        classes=np.array([0, 1]),
        y=y_fit,
    )
    class_weights = {0: weights[0], 1: weights[1]}
    real_weights = np.array(
        [class_weights[v] for v in y_fit], dtype="float32"
    )

    def make_model():
        model = Sequential([
            Input(shape=(X_fit.shape[1],)),
            Dense(16, activation="relu"),
            Dropout(0.1),
            Dense(8, activation="relu"),
            Dense(1, activation="sigmoid"),
        ])
        model.compile(
            optimizer="adam",
            loss="binary_crossentropy",
        )
        return model

    def train(model, X_data, y_data, sample_weights):
        model.fit(
            X_data,
            y_data,
            sample_weight=sample_weights,
            validation_data=(X_val, y_val),
            epochs=50,
            batch_size=32,
            callbacks=[
                EarlyStopping(
                    monitor="val_loss",
                    patience=8,
                    restore_best_weights=True,
                )
            ],
            verbose=0,
        )

    # ① 라벨 데이터만 학습
    tf.keras.utils.set_random_seed(42)
    base = make_model()
    train(base, X_fit, y_fit, real_weights)

    print(f"\n[{product.upper()} 작은 Keras: 라벨만]")
    base_prob = base.predict(X_test, verbose=0).ravel()
    evaluate_model(y_test, (base_prob >= 0.5).astype(int))
    print("확률 기반 ROC-AUC:", roc_auc_score(y_test, base_prob))

    # ② 확신도 0.95 이상인 미라벨 데이터 중 최대 10% 선택
    unlabeled_prob = base.predict(X_unlabeled, verbose=0).ravel()
    confidence = np.maximum(unlabeled_prob, 1 - unlabeled_prob)
    candidates = np.flatnonzero(confidence >= 0.95)
    max_count = max(1, int(len(X_unlabeled) * 0.10))
    selected = candidates[np.argsort(confidence[candidates])[::-1][:max_count]]

    if len(selected) == 0:
        print("선택된 임시 라벨이 없어 준지도 재학습을 건너뜁니다.")
        return base

    X_pseudo = X_unlabeled[selected]
    y_pseudo = (unlabeled_prob[selected] >= 0.5).astype(int)
    print("임시 라벨 개수:", pd.Series(y_pseudo).value_counts().to_dict())

    pseudo_weights = np.array(
        [class_weights[v] * 0.2 for v in y_pseudo],
        dtype="float32",
    )

    # ③ 동일한 초기 시드로 새 모델을 만들어 재학습
    tf.keras.utils.set_random_seed(42)
    semi = make_model()
    train(
        semi,
        np.concatenate([X_fit, X_pseudo]),
        np.concatenate([y_fit, y_pseudo]),
        np.concatenate([real_weights, pseudo_weights]),
    )

    print(f"\n[{product.upper()} 작은 Keras: 준지도]")
    semi_prob = semi.predict(X_test, verbose=0).ravel()
    evaluate_model(y_test, (semi_prob >= 0.5).astype(int))
    print("확률 기반 ROC-AUC:", roc_auc_score(y_test, semi_prob))
    return semi

small_keras_cn7 = run_small_keras_semi("cn7")
# small_keras_rg3 = run_small_keras_semi("rg3")

CN7: 라벨 학습 847건, 평가 364건, 미라벨 35239건

[CN7 작은 Keras: 라벨만]
혼동행렬:
 [[355   4]
 [  3   2]]
정확도: 0.9807692307692307
정밀도: 0.3333333333333333
재현율: 0.4
F1: 0.36363636363636365
ROC-AUC: 0.6944289693593314
확률 기반 ROC-AUC: 0.9537604456824512
임시 라벨 개수: {0: 3523}

[CN7 작은 Keras: 준지도]
혼동행렬:
 [[356   3]
 [  3   2]]
정확도: 0.9835164835164835
정밀도: 0.4
재현율: 0.4
F1: 0.4
ROC-AUC: 0.6958217270194986
확률 기반 ROC-AUC: 0.9571030640668524


In [15]:
## 준지도학습 중 라벨링과 테스트 모델을 다르게 함.

In [16]:
# 기존 공통 코드 load_product_data(), evaluate_model() 실행 후 사용

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import fbeta_score, average_precision_score


def run_cross_model_semi(product="cn7"):
    # --------------------------------------------------
    # 1. 기존 공통 함수로 데이터 불러오기
    # --------------------------------------------------
    X_train_all, y_train_all, X_test, y_test, X_unlabeled = (
        load_product_data(product)
    )

    y_train_all = y_train_all.to_numpy()
    y_test = y_test.to_numpy()

    # 라벨 학습 데이터 안에서 검증 데이터 분리
    X_fit, X_val, y_fit, y_val = train_test_split(
        X_train_all,
        y_train_all,
        test_size=0.25,
        stratify=y_train_all,
        random_state=42,
    )

    # SVC를 위해 스케일링. 학습 데이터로만 fit
    scaler = StandardScaler()
    X_fit = scaler.fit_transform(X_fit).astype("float32")
    X_val = scaler.transform(X_val).astype("float32")
    X_test = scaler.transform(X_test).astype("float32")
    X_unlabeled = scaler.transform(X_unlabeled).astype("float32")

    # 검증·평가 데이터와 공정 변수값이 완전히 같은 미라벨 행은 제외
    held_out = pd.DataFrame(
        np.concatenate([X_val, X_test])
    )
    unlabeled_frame = pd.DataFrame(X_unlabeled)

    held_out_hash = set(
        pd.util.hash_pandas_object(
            held_out, index=False
        ).to_numpy()
    )
    unlabeled_hash = pd.util.hash_pandas_object(
        unlabeled_frame, index=False
    ).to_numpy()

    keep = ~np.isin(unlabeled_hash, list(held_out_hash))
    X_unlabeled = X_unlabeled[keep]

    print(
        f"\n{product.upper()} | 학습 불량 {sum(y_fit == 1)}건, "
        f"검증 불량 {sum(y_val == 1)}건, "
        f"평가 불량 {sum(y_test == 1)}건"
    )
    print("임시 라벨 후보:", len(X_unlabeled), "건")

    # 가중치가 지나치게 커지는 것을 막기 위한 실험 설정
    negative_count = sum(y_fit == 0)
    positive_count = sum(y_fit == 1)
    positive_weight = min(
        negative_count / positive_count, 20
    )

    # --------------------------------------------------
    # 2. XGBoost가 임시 라벨 부착
    # --------------------------------------------------
    teacher = XGBClassifier(
        n_estimators=100,
        max_depth=2,
        learning_rate=0.05,
        min_child_weight=3,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        scale_pos_weight=positive_weight,
        random_state=42,
        n_jobs=-1,
    )

    teacher.fit(X_fit, y_fit)

    teacher_prob = teacher.predict_proba(
        X_unlabeled
    )[:, 1]

    # 확신도 0.95 이상인 것만 선택
    positive_candidates = np.flatnonzero(
        teacher_prob >= 0.95
    )
    negative_candidates = np.flatnonzero(
        teacher_prob <= 0.05
    )

    # 임시 라벨이 공식 라벨을 압도하지 않도록 개수 제한
    max_positive = 2 * positive_count
    max_negative = 4 * positive_count

    selected_positive = positive_candidates[
        np.argsort(
            teacher_prob[positive_candidates]
        )[::-1][:max_positive]
    ]
    selected_negative = negative_candidates[
        np.argsort(
            teacher_prob[negative_candidates]
        )[:max_negative]
    ]

    selected = np.concatenate([
        selected_negative,
        selected_positive,
    ])

    X_pseudo = X_unlabeled[selected]
    y_pseudo = np.concatenate([
        np.zeros(len(selected_negative), dtype=int),
        np.ones(len(selected_positive), dtype=int),
    ])

    print(
        "XGBoost 임시 라벨 | "
        f"양품 {len(selected_negative)}건, "
        f"불량 {len(selected_positive)}건"
    )

    # --------------------------------------------------
    # 3. 검증 데이터로 판정 기준 선택
    # --------------------------------------------------
    def select_threshold(model):
        val_prob = model.predict_proba(X_val)[:, 1]

        candidates = np.linspace(0.10, 0.90, 17)
        scores = []

        for threshold in candidates:
            pred = (val_prob >= threshold).astype(int)
            f2 = fbeta_score(
                y_val, pred, beta=2, zero_division=0
            )
            fp = np.sum((y_val == 0) & (pred == 1))
            scores.append((f2, -fp, threshold))

        # F2가 높고, 동점이면 양품 오탐이 적은 기준 선택
        best = max(scores)
        return float(best[2])

    def show_result(name, model):
        threshold = select_threshold(model)
        test_prob = model.predict_proba(X_test)[:, 1]
        test_pred = (test_prob >= threshold).astype(int)

        print(f"\n[{product.upper()} {name}]")
        print("검증 데이터에서 선택한 판정 기준:", threshold)
        evaluate_model(y_test, test_pred)
        print(
            "확률 기반 ROC-AUC:",
            roc_auc_score(y_test, test_prob)
        )
        print(
            "PR-AUC(AP):",
            average_precision_score(y_test, test_prob)
        )

    # --------------------------------------------------
    # 4. CatBoost: 공식 라벨만 / 준지도 비교
    # --------------------------------------------------
    def make_catboost():
        return CatBoostClassifier(
            iterations=200,
            depth=3,
            learning_rate=0.03,
            loss_function="Logloss",
            class_weights=[1.0, positive_weight],
            random_seed=42,
            verbose=False,
        )

    cat_base = make_catboost()
    cat_base.fit(X_fit, y_fit)
    show_result("CatBoost: 공식 라벨만", cat_base)

    if len(y_pseudo) > 0:
        cat_semi = make_catboost()
        cat_semi.fit(
            np.concatenate([X_fit, X_pseudo]),
            np.concatenate([y_fit, y_pseudo]),
            sample_weight=np.concatenate([
                np.ones(len(y_fit)),
                np.full(len(y_pseudo), 0.2),
            ]),
        )
        show_result(
            "XGBoost 임시 라벨 → CatBoost", cat_semi
        )

    # --------------------------------------------------
    # 5. SVC: 공식 라벨만 / 준지도 비교
    # --------------------------------------------------
    def make_svc():
        return SVC(
            C=1.0,
            kernel="rbf",
            gamma="scale",
            class_weight={
                0: 1.0,
                1: positive_weight,
            },
            probability=True,
            random_state=42,
        )

    svc_base = make_svc()
    svc_base.fit(X_fit, y_fit)
    show_result("SVC: 공식 라벨만", svc_base)

    if len(y_pseudo) > 0:
        svc_semi = make_svc()
        svc_semi.fit(
            np.concatenate([X_fit, X_pseudo]),
            np.concatenate([y_fit, y_pseudo]),
            sample_weight=np.concatenate([
                np.ones(len(y_fit)),
                np.full(len(y_pseudo), 0.2),
            ]),
        )
        show_result(
            "XGBoost 임시 라벨 → SVC", svc_semi
        )

    return {
        "teacher": teacher,
        "catboost_baseline": cat_base,
        "catboost_semi": cat_semi if len(y_pseudo) else None,
        "svc_baseline": svc_base,
        "svc_semi": svc_semi if len(y_pseudo) else None,
    }


# CN7 실행
cn7_models = run_cross_model_semi("cn7")

# RG3 실행 시 별도로 실행
# rg3_models = run_cross_model_semi("rg3")

CN7: 라벨 학습 847건, 평가 364건, 미라벨 35239건

CN7 | 학습 불량 9건, 검증 불량 3건, 평가 불량 5건
임시 라벨 후보: 35239 건
XGBoost 임시 라벨 | 양품 36건, 불량 0건

[CN7 CatBoost: 공식 라벨만]
검증 데이터에서 선택한 판정 기준: 0.2
혼동행렬:
 [[356   3]
 [  3   2]]
정확도: 0.9835164835164835
정밀도: 0.4
재현율: 0.4
F1: 0.4
ROC-AUC: 0.6958217270194986
확률 기반 ROC-AUC: 0.9805013927576601
PR-AUC(AP): 0.562878787878788

[CN7 XGBoost 임시 라벨 → CatBoost]
검증 데이터에서 선택한 판정 기준: 0.25
혼동행렬:
 [[355   4]
 [  3   2]]
정확도: 0.9807692307692307
정밀도: 0.3333333333333333
재현율: 0.4
F1: 0.36363636363636365
ROC-AUC: 0.6944289693593314
확률 기반 ROC-AUC: 0.9715877437325906
PR-AUC(AP): 0.550297619047619

[CN7 SVC: 공식 라벨만]
검증 데이터에서 선택한 판정 기준: 0.9
혼동행렬:
 [[359   0]
 [  5   0]]
정확도: 0.9862637362637363
정밀도: 0.0
재현율: 0.0
F1: 0.0
ROC-AUC: 0.5
확률 기반 ROC-AUC: 0.8434540389972144
PR-AUC(AP): 0.3416666666666667

[CN7 XGBoost 임시 라벨 → SVC]
검증 데이터에서 선택한 판정 기준: 0.9
혼동행렬:
 [[359   0]
 [  5   0]]
정확도: 0.9862637362637363
정밀도: 0.0
재현율: 0.0
F1: 0.0
ROC-AUC: 0.5
확률 기반 ROC-AUC: 0.8646239554317549
PR-AUC(AP): 0.506009

In [20]:
from xgboost import XGBClassifier

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import fbeta_score

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam


def run_small_keras_to_xgboost(product="cn7"):
    # --------------------------------------------------
    # 1. 기존 공통 함수로 데이터 준비
    # --------------------------------------------------
    X_train_all, y_train_all, X_test, y_test, X_unlabeled = (
        load_product_data(product)
    )

    y_train_all = y_train_all.to_numpy()
    y_test = y_test.to_numpy()

    # 평가 데이터는 건드리지 않고, 학습 데이터에서 검증용 분리
    X_fit, X_val, y_fit, y_val = train_test_split(
        X_train_all,
        y_train_all,
        test_size=0.25,
        stratify=y_train_all,
        random_state=42,
    )

    scaler = StandardScaler()
    X_fit = scaler.fit_transform(X_fit).astype("float32")
    X_val = scaler.transform(X_val).astype("float32")
    X_test = scaler.transform(X_test).astype("float32")
    X_unlabeled = scaler.transform(X_unlabeled).astype("float32")

    n_pos = int(np.sum(y_fit == 1))
    n_neg = int(np.sum(y_fit == 0))
    positive_weight = min(n_neg / n_pos, 20.0)

    print(
        f"{product.upper()} | 학습 불량 {n_pos}건, "
        f"검증 불량 {sum(y_val == 1)}건, "
        f"평가 불량 {sum(y_test == 1)}건"
    )

    # --------------------------------------------------
    # 2. 임시 라벨을 붙일 작은 Keras 모델
    # --------------------------------------------------
    tf.keras.utils.set_random_seed(42)

    teacher = Sequential([
        Input(shape=(X_fit.shape[1],)),
        Dense(32, activation="relu"),
        Dropout(0.1),
        Dense(16, activation="relu"),
        Dense(1, activation="sigmoid"),
    ])

    teacher.compile(
        optimizer=Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
        ],
    )

    # 실제 라벨과 현재까지 추가한 임시 라벨
    X_with_label = X_fit.copy()
    y_with_label = y_fit.copy()

    real_weights = np.where(
        y_fit == 1, positive_weight, 1.0
    ).astype("float32")
    sample_weights = real_weights.copy()

    without_label = X_unlabeled.copy()
    original_unlabeled_count = len(without_label)

    # PDF의 unlabeled_usage=90:
    # 처음 미라벨 데이터의 10%가 남으면 종료
    target_remaining = int(original_unlabeled_count * 0.10)

    round_number = 0
    total_pseudo_normal = 0
    total_pseudo_defect = 0

    # --------------------------------------------------
    # 3. PDF처럼 회차별 학습 → 임시 라벨 추가
    # --------------------------------------------------
    while len(without_label) > target_remaining:
        round_number += 1

        history = teacher.fit(
            X_with_label,
            y_with_label,
            sample_weight=sample_weights,
            validation_data=(X_val, y_val),
            batch_size=32,
            epochs=100,  # PDF: 최대 100 epoch
            callbacks=[
                EarlyStopping(
                    monitor="val_loss",
                    patience=10,  # PDF: 10 epoch 대기
                    restore_best_weights=True,
                )
            ],
            verbose=0,
        )

        probabilities = teacher.predict(
            without_label, verbose=0
        ).ravel()

        # PDF처럼 두 클래스 중 높은 확률을 확신도로 사용
        confidence = np.maximum(
            probabilities, 1 - probabilities
        )

        # 매 회차 남은 데이터의 10%를 선택
        count = max(1, int(len(without_label) * 0.10))
        count = min(
            count,
            len(without_label) - target_remaining,
        )

        selected = np.argsort(confidence)[::-1][:count]
        remaining_mask = np.ones(
            len(without_label), dtype=bool
        )
        remaining_mask[selected] = False

        X_pseudo = without_label[selected]
        y_pseudo = (
            probabilities[selected] >= 0.5
        ).astype(int)

        # 임시 라벨은 실제 라벨보다 낮은 가중치로 학습
        pseudo_weights = (
            np.where(
                y_pseudo == 1,
                positive_weight,
                1.0,
            ) * 0.2
        ).astype("float32")

        X_with_label = np.concatenate([
            X_with_label, X_pseudo
        ])
        y_with_label = np.concatenate([
            y_with_label, y_pseudo
        ])
        sample_weights = np.concatenate([
            sample_weights, pseudo_weights
        ])

        without_label = without_label[remaining_mask]

        added_normal = int(np.sum(y_pseudo == 0))
        added_defect = int(np.sum(y_pseudo == 1))
        total_pseudo_normal += added_normal
        total_pseudo_defect += added_defect

        print(
            f"{round_number}회차 | "
            f"실제 학습 {len(history.history['loss'])} epoch | "
            f"임시 양품 {added_normal}건, "
            f"임시 불량 {added_defect}건 | "
            f"남은 미라벨 {len(without_label)}건"
        )

    print(
        f"\n총 {round_number}회차 | "
        f"임시 양품 {total_pseudo_normal}건, "
        f"임시 불량 {total_pseudo_defect}건"
    )

    # --------------------------------------------------
    # 4. 최종 판정용 XGBoost:
    #    공식 라벨만 / 임시 라벨 추가 결과 비교
    # --------------------------------------------------
    def make_final_model():
        return XGBClassifier(
            n_estimators=100,
            max_depth=2,
            learning_rate=0.05,
            min_child_weight=3,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=positive_weight,
            random_state=42,
            n_jobs=-1,
        )

    def choose_threshold(model):
        val_prob = model.predict_proba(X_val)[:, 1]
        candidates = np.linspace(0.1, 0.9, 17)

        scores = []
        for threshold in candidates:
            prediction = (
                val_prob >= threshold
            ).astype(int)

            f2 = fbeta_score(
                y_val, prediction,
                beta=2,
                zero_division=0,
            )
            false_positives = np.sum(
                (y_val == 0) & (prediction == 1)
            )
            scores.append(
                (f2, -false_positives, threshold)
            )

        return float(max(scores)[2])

    def show_result(name, model):
        threshold = choose_threshold(model)
        test_prob = model.predict_proba(X_test)[:, 1]
        test_pred = (
            test_prob >= threshold
        ).astype(int)

        print(f"\n[{product.upper()} {name}]")
        print("검증 데이터에서 정한 판정 기준:", threshold)
        evaluate_model(y_test, test_pred)
        print(
            "확률 기반 ROC-AUC:",
            roc_auc_score(y_test, test_prob)
        )

    # 기준 모델: 실제 라벨만
    baseline = make_final_model()
    baseline.fit(X_fit, y_fit)
    show_result("XGBoost: 공식 라벨만", baseline)

    # 준지도 모델: 작은 Keras가 붙인 임시 라벨 추가
    semi_model = make_final_model()
    semi_model.fit(
        X_with_label,
        y_with_label,
        sample_weight=sample_weights,
    )
    show_result(
        "작은 Keras 임시 라벨 → XGBoost",
        semi_model,
    )

    return teacher, baseline, semi_model


# CN7 실행
cn7_teacher, cn7_baseline, cn7_semi = (
    run_small_keras_to_xgboost("cn7")
)

# RG3는 별도로 실행
# rg3_teacher, rg3_baseline, rg3_semi = (
#     run_small_keras_to_xgboost("rg3")
# )

CN7: 라벨 학습 847건, 평가 364건, 미라벨 35239건
CN7 | 학습 불량 9건, 검증 불량 3건, 평가 불량 5건
1회차 | 실제 학습 40 epoch | 임시 양품 3523건, 임시 불량 0건 | 남은 미라벨 31716건
2회차 | 실제 학습 11 epoch | 임시 양품 3171건, 임시 불량 0건 | 남은 미라벨 28545건
3회차 | 실제 학습 12 epoch | 임시 양품 2854건, 임시 불량 0건 | 남은 미라벨 25691건
4회차 | 실제 학습 11 epoch | 임시 양품 2569건, 임시 불량 0건 | 남은 미라벨 23122건
5회차 | 실제 학습 13 epoch | 임시 양품 2312건, 임시 불량 0건 | 남은 미라벨 20810건
6회차 | 실제 학습 14 epoch | 임시 양품 2081건, 임시 불량 0건 | 남은 미라벨 18729건
7회차 | 실제 학습 11 epoch | 임시 양품 1872건, 임시 불량 0건 | 남은 미라벨 16857건
8회차 | 실제 학습 11 epoch | 임시 양품 1077건, 임시 불량 608건 | 남은 미라벨 15172건
9회차 | 실제 학습 14 epoch | 임시 양품 0건, 임시 불량 1517건 | 남은 미라벨 13655건
10회차 | 실제 학습 12 epoch | 임시 양품 0건, 임시 불량 1365건 | 남은 미라벨 12290건
11회차 | 실제 학습 14 epoch | 임시 양품 0건, 임시 불량 1229건 | 남은 미라벨 11061건
12회차 | 실제 학습 11 epoch | 임시 양품 0건, 임시 불량 1106건 | 남은 미라벨 9955건
13회차 | 실제 학습 19 epoch | 임시 양품 0건, 임시 불량 995건 | 남은 미라벨 8960건
14회차 | 실제 학습 11 epoch | 임시 양품 0건, 임시 불량 896건 | 남은 미라벨 8064건
15회차 | 실제 학습 19 epoch | 임시 양품 0건, 임시 불량 806건 | 남은 미라벨 7258건
16회차 | 실제 학습

In [19]:
from xgboost import XGBClassifier

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import fbeta_score

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam


def run_small_keras_to_xgboost(product="cn7"):
    # --------------------------------------------------
    # 1. 기존 공통 함수로 데이터 준비
    # --------------------------------------------------
    X_train_all, y_train_all, X_test, y_test, X_unlabeled = (
        load_product_data(product)
    )

    y_train_all = y_train_all.to_numpy()
    y_test = y_test.to_numpy()

    # 평가 데이터는 건드리지 않고, 학습 데이터에서 검증용 분리
    X_fit, X_val, y_fit, y_val = train_test_split(
        X_train_all,
        y_train_all,
        test_size=0.25,     
        stratify=y_train_all,
        random_state=42,
    )

    scaler = StandardScaler()
    X_fit = scaler.fit_transform(X_fit).astype("float32")
    X_val = scaler.transform(X_val).astype("float32")
    X_test = scaler.transform(X_test).astype("float32")
    X_unlabeled = scaler.transform(X_unlabeled).astype("float32")

    n_pos = int(np.sum(y_fit == 1))
    n_neg = int(np.sum(y_fit == 0))
    positive_weight = min(n_neg / n_pos, 20.0)

    print(
        f"{product.upper()} | 학습 불량 {n_pos}건, "
        f"검증 불량 {sum(y_val == 1)}건, "
        f"평가 불량 {sum(y_test == 1)}건"
    )

    # --------------------------------------------------
    # 2. 임시 라벨을 붙일 작은 Keras 모델
    # --------------------------------------------------
    tf.keras.utils.set_random_seed(42)

    teacher = Sequential([
        Input(shape=(X_fit.shape[1],)),
        Dense(32, activation="relu"),
        Dropout(0.1),
        Dense(16, activation="relu"),
        Dense(1, activation="sigmoid"),
    ])

    teacher.compile(
        optimizer=Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
        ],
    )

    # 실제 라벨과 현재까지 추가한 임시 라벨
    X_with_label = X_fit.copy()
    y_with_label = y_fit.copy()

    real_weights = np.where(
        y_fit == 1, positive_weight, 1.0
    ).astype("float32")
    sample_weights = real_weights.copy()

    without_label = X_unlabeled.copy()
    original_unlabeled_count = len(without_label)

    # PDF의 unlabeled_usage=90:
    # 처음 미라벨 데이터의 10%가 남으면 종료
    target_remaining = int(original_unlabeled_count * 0.10)

    round_number = 0
    total_pseudo_normal = 0
    total_pseudo_defect = 0

    # --------------------------------------------------
    # 3. PDF처럼 회차별 학습 → 임시 라벨 추가
    # --------------------------------------------------
    while len(without_label) > target_remaining:
        round_number += 1

        history = teacher.fit(
            X_with_label,
            y_with_label,
            sample_weight=sample_weights,
            validation_data=(X_val, y_val),
            batch_size=32,
            epochs=100,  # PDF: 최대 100 epoch
            callbacks=[
                EarlyStopping(
                    monitor="val_loss",
                    patience=10,  # PDF: 10 epoch 대기
                    restore_best_weights=True,
                )
            ],
            verbose=0,
        )

        probabilities = teacher.predict(
            without_label, verbose=0
        ).ravel()

        # PDF처럼 두 클래스 중 높은 확률을 확신도로 사용
        confidence = np.maximum(
            probabilities, 1 - probabilities
        )

        # 매 회차 남은 데이터의 10%를 선택
        count = max(1, int(len(without_label) * 0.10))
        count = min(
            count,
            len(without_label) - target_remaining,
        )

        selected = np.argsort(confidence)[::-1][:count]
        remaining_mask = np.ones(
            len(without_label), dtype=bool
        )
        remaining_mask[selected] = False

        X_pseudo = without_label[selected]
        y_pseudo = (
            probabilities[selected] >= 0.5
        ).astype(int)

        # 임시 라벨은 실제 라벨보다 낮은 가중치로 학습
        pseudo_weights = (
            np.where(
                y_pseudo == 1,
                positive_weight,
                1.0,
            ) * 0.2
        ).astype("float32")

        X_with_label = np.concatenate([
            X_with_label, X_pseudo
        ])
        y_with_label = np.concatenate([
            y_with_label, y_pseudo
        ])
        sample_weights = np.concatenate([
            sample_weights, pseudo_weights
        ])

        without_label = without_label[remaining_mask]

        added_normal = int(np.sum(y_pseudo == 0))
        added_defect = int(np.sum(y_pseudo == 1))
        total_pseudo_normal += added_normal
        total_pseudo_defect += added_defect

        print(
            f"{round_number}회차 | "
            f"실제 학습 {len(history.history['loss'])} epoch | "
            f"임시 양품 {added_normal}건, "
            f"임시 불량 {added_defect}건 | "
            f"남은 미라벨 {len(without_label)}건"
        )

    print(
        f"\n총 {round_number}회차 | "
        f"임시 양품 {total_pseudo_normal}건, "
        f"임시 불량 {total_pseudo_defect}건"
    )

    # --------------------------------------------------
    # 4. 최종 판정용 XGBoost:
    #    공식 라벨만 / 임시 라벨 추가 결과 비교
    # --------------------------------------------------
    def make_final_model():
        return XGBClassifier(
            n_estimators=100,
            max_depth=2,
            learning_rate=0.05,
            min_child_weight=3,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=positive_weight,
            random_state=42,
            n_jobs=-1,
        )

    def choose_threshold(model):
        val_prob = model.predict_proba(X_val)[:, 1]
        candidates = np.linspace(0.15, 0.85, 17)

        scores = []
        for threshold in candidates:
            prediction = (
                val_prob >= threshold
            ).astype(int)

            f2 = fbeta_score(
                y_val, prediction,
                beta=2,
                zero_division=0,
            )
            false_positives = np.sum(
                (y_val == 0) & (prediction == 1)
            )
            scores.append(
                (f2, -false_positives, threshold)
            )

        return float(max(scores)[2])

    def show_result(name, model):
        threshold = choose_threshold(model)
        test_prob = model.predict_proba(X_test)[:, 1]
        test_pred = (
            test_prob >= threshold
        ).astype(int)

        print(f"\n[{product.upper()} {name}]")
        print("검증 데이터에서 정한 판정 기준:", threshold)
        evaluate_model(y_test, test_pred)
        print(
            "확률 기반 ROC-AUC:",
            roc_auc_score(y_test, test_prob)
        )

    # 기준 모델: 실제 라벨만
    baseline = make_final_model()
    baseline.fit(X_fit, y_fit)
    show_result("XGBoost: 공식 라벨만", baseline)

    # 준지도 모델: 작은 Keras가 붙인 임시 라벨 추가
    semi_model = make_final_model()
    semi_model.fit(
        X_with_label,
        y_with_label,
        sample_weight=sample_weights,
    )
    show_result(
        "작은 Keras 임시 라벨 → XGBoost",
        semi_model,
    )

    return teacher, baseline, semi_model


# CN7 실행
cn7_teacher, cn7_baseline, cn7_semi = (
    run_small_keras_to_xgboost("cn7")
)

# RG3는 별도로 실행
# rg3_teacher, rg3_baseline, rg3_semi = (
#     run_small_keras_to_xgboost("rg3")
# )

CN7: 라벨 학습 847건, 평가 364건, 미라벨 35239건
CN7 | 학습 불량 9건, 검증 불량 3건, 평가 불량 5건
1회차 | 실제 학습 40 epoch | 임시 양품 3523건, 임시 불량 0건 | 남은 미라벨 31716건
2회차 | 실제 학습 11 epoch | 임시 양품 3171건, 임시 불량 0건 | 남은 미라벨 28545건
3회차 | 실제 학습 12 epoch | 임시 양품 2854건, 임시 불량 0건 | 남은 미라벨 25691건
4회차 | 실제 학습 11 epoch | 임시 양품 2569건, 임시 불량 0건 | 남은 미라벨 23122건
5회차 | 실제 학습 13 epoch | 임시 양품 2312건, 임시 불량 0건 | 남은 미라벨 20810건
6회차 | 실제 학습 14 epoch | 임시 양품 2081건, 임시 불량 0건 | 남은 미라벨 18729건
7회차 | 실제 학습 11 epoch | 임시 양품 1872건, 임시 불량 0건 | 남은 미라벨 16857건
8회차 | 실제 학습 11 epoch | 임시 양품 1077건, 임시 불량 608건 | 남은 미라벨 15172건
9회차 | 실제 학습 14 epoch | 임시 양품 0건, 임시 불량 1517건 | 남은 미라벨 13655건
10회차 | 실제 학습 12 epoch | 임시 양품 0건, 임시 불량 1365건 | 남은 미라벨 12290건
11회차 | 실제 학습 14 epoch | 임시 양품 0건, 임시 불량 1229건 | 남은 미라벨 11061건
12회차 | 실제 학습 11 epoch | 임시 양품 0건, 임시 불량 1106건 | 남은 미라벨 9955건
13회차 | 실제 학습 19 epoch | 임시 양품 0건, 임시 불량 995건 | 남은 미라벨 8960건
14회차 | 실제 학습 11 epoch | 임시 양품 0건, 임시 불량 896건 | 남은 미라벨 8064건
15회차 | 실제 학습 19 epoch | 임시 양품 0건, 임시 불량 806건 | 남은 미라벨 7258건
16회차 | 실제 학습

In [21]:
from xgboost import XGBClassifier

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import fbeta_score

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam


def run_small_keras_to_xgboost(product="cn7"):
    # --------------------------------------------------
    # 1. 기존 공통 함수로 데이터 준비
    # --------------------------------------------------
    X_train_all, y_train_all, X_test, y_test, X_unlabeled = (
        load_product_data(product)
    )

    y_train_all = y_train_all.to_numpy()
    y_test = y_test.to_numpy()

    # 평가 데이터는 건드리지 않고, 학습 데이터에서 검증용 분리
    X_fit, X_val, y_fit, y_val = train_test_split(
        X_train_all,
        y_train_all,
        test_size=0.2,     
        stratify=y_train_all,
        random_state=42,
    )

    scaler = StandardScaler()
    X_fit = scaler.fit_transform(X_fit).astype("float32")
    X_val = scaler.transform(X_val).astype("float32")
    X_test = scaler.transform(X_test).astype("float32")
    X_unlabeled = scaler.transform(X_unlabeled).astype("float32")

    n_pos = int(np.sum(y_fit == 1))
    n_neg = int(np.sum(y_fit == 0))
    positive_weight = min(n_neg / n_pos, 20.0)

    print(
        f"{product.upper()} | 학습 불량 {n_pos}건, "
        f"검증 불량 {sum(y_val == 1)}건, "
        f"평가 불량 {sum(y_test == 1)}건"
    )

    # --------------------------------------------------
    # 2. 임시 라벨을 붙일 작은 Keras 모델
    # --------------------------------------------------
    tf.keras.utils.set_random_seed(42)

    teacher = Sequential([
        Input(shape=(X_fit.shape[1],)),
        Dense(32, activation="relu"),
        Dropout(0.1),
        Dense(16, activation="relu"),
        Dense(1, activation="sigmoid"),
    ])

    teacher.compile(
        optimizer=Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
        ],
    )

    # 실제 라벨과 현재까지 추가한 임시 라벨
    X_with_label = X_fit.copy()
    y_with_label = y_fit.copy()

    real_weights = np.where(
        y_fit == 1, positive_weight, 1.0
    ).astype("float32")
    sample_weights = real_weights.copy()

    without_label = X_unlabeled.copy()
    original_unlabeled_count = len(without_label)

    # PDF의 unlabeled_usage=90:
    # 처음 미라벨 데이터의 10%가 남으면 종료
    target_remaining = int(original_unlabeled_count * 0.10)

    round_number = 0
    total_pseudo_normal = 0
    total_pseudo_defect = 0

    # --------------------------------------------------
    # 3. PDF처럼 회차별 학습 → 임시 라벨 추가
    # --------------------------------------------------
    while len(without_label) > target_remaining:
        round_number += 1

        history = teacher.fit(
            X_with_label,
            y_with_label,
            sample_weight=sample_weights,
            validation_data=(X_val, y_val),
            batch_size=32,
            epochs=100,  # PDF: 최대 100 epoch
            callbacks=[
                EarlyStopping(
                    monitor="val_loss",
                    patience=10,  # PDF: 10 epoch 대기
                    restore_best_weights=True,
                )
            ],
            verbose=0,
        )

        probabilities = teacher.predict(
            without_label, verbose=0
        ).ravel()

        # PDF처럼 두 클래스 중 높은 확률을 확신도로 사용
        confidence = np.maximum(
            probabilities, 1 - probabilities
        )

        # 매 회차 남은 데이터의 10%를 선택
        count = max(1, int(len(without_label) * 0.10))
        count = min(
            count,
            len(without_label) - target_remaining,
        )

        selected = np.argsort(confidence)[::-1][:count]
        remaining_mask = np.ones(
            len(without_label), dtype=bool
        )
        remaining_mask[selected] = False

        X_pseudo = without_label[selected]
        y_pseudo = (
            probabilities[selected] >= 0.5
        ).astype(int)

        # 임시 라벨은 실제 라벨보다 낮은 가중치로 학습
        pseudo_weights = (
            np.where(
                y_pseudo == 1,
                positive_weight,
                1.0,
            ) * 0.2
        ).astype("float32")

        X_with_label = np.concatenate([
            X_with_label, X_pseudo
        ])
        y_with_label = np.concatenate([
            y_with_label, y_pseudo
        ])
        sample_weights = np.concatenate([
            sample_weights, pseudo_weights
        ])

        without_label = without_label[remaining_mask]

        added_normal = int(np.sum(y_pseudo == 0))
        added_defect = int(np.sum(y_pseudo == 1))
        total_pseudo_normal += added_normal
        total_pseudo_defect += added_defect

        print(
            f"{round_number}회차 | "
            f"실제 학습 {len(history.history['loss'])} epoch | "
            f"임시 양품 {added_normal}건, "
            f"임시 불량 {added_defect}건 | "
            f"남은 미라벨 {len(without_label)}건"
        )

    print(
        f"\n총 {round_number}회차 | "
        f"임시 양품 {total_pseudo_normal}건, "
        f"임시 불량 {total_pseudo_defect}건"
    )

    # --------------------------------------------------
    # 4. 최종 판정용 XGBoost:
    #    공식 라벨만 / 임시 라벨 추가 결과 비교
    # --------------------------------------------------
    def make_final_model():
        return XGBClassifier(
            n_estimators=100,
            max_depth=2,
            learning_rate=0.05,
            min_child_weight=3,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=positive_weight,
            random_state=42,
            n_jobs=-1,
        )

    def choose_threshold(model):
        val_prob = model.predict_proba(X_val)[:, 1]
        candidates = np.linspace(0.15, 0.85, 17)

        scores = []
        for threshold in candidates:
            prediction = (
                val_prob >= threshold
            ).astype(int)

            f2 = fbeta_score(
                y_val, prediction,
                beta=2,
                zero_division=0,
            )
            false_positives = np.sum(
                (y_val == 0) & (prediction == 1)
            )
            scores.append(
                (f2, -false_positives, threshold)
            )

        return float(max(scores)[2])

    def show_result(name, model):
        threshold = choose_threshold(model)
        test_prob = model.predict_proba(X_test)[:, 1]
        test_pred = (
            test_prob >= threshold
        ).astype(int)

        print(f"\n[{product.upper()} {name}]")
        print("검증 데이터에서 정한 판정 기준:", threshold)
        evaluate_model(y_test, test_pred)
        print(
            "확률 기반 ROC-AUC:",
            roc_auc_score(y_test, test_prob)
        )

    # 기준 모델: 실제 라벨만
    baseline = make_final_model()
    baseline.fit(X_fit, y_fit)
    show_result("XGBoost: 공식 라벨만", baseline)

    # 준지도 모델: 작은 Keras가 붙인 임시 라벨 추가
    semi_model = make_final_model()
    semi_model.fit(
        X_with_label,
        y_with_label,
        sample_weight=sample_weights,
    )
    show_result(
        "작은 Keras 임시 라벨 → XGBoost",
        semi_model,
    )

    return teacher, baseline, semi_model


# CN7 실행
cn7_teacher, cn7_baseline, cn7_semi = (
    run_small_keras_to_xgboost("cn7")
)

# RG3는 별도로 실행
# rg3_teacher, rg3_baseline, rg3_semi = (
#     run_small_keras_to_xgboost("rg3")
# )

CN7: 라벨 학습 847건, 평가 364건, 미라벨 35239건
CN7 | 학습 불량 10건, 검증 불량 2건, 평가 불량 5건
1회차 | 실제 학습 37 epoch | 임시 양품 3523건, 임시 불량 0건 | 남은 미라벨 31716건
2회차 | 실제 학습 24 epoch | 임시 양품 3171건, 임시 불량 0건 | 남은 미라벨 28545건
3회차 | 실제 학습 11 epoch | 임시 양품 2854건, 임시 불량 0건 | 남은 미라벨 25691건
4회차 | 실제 학습 11 epoch | 임시 양품 2569건, 임시 불량 0건 | 남은 미라벨 23122건
5회차 | 실제 학습 15 epoch | 임시 양품 2312건, 임시 불량 0건 | 남은 미라벨 20810건
6회차 | 실제 학습 19 epoch | 임시 양품 2081건, 임시 불량 0건 | 남은 미라벨 18729건
7회차 | 실제 학습 16 epoch | 임시 양품 1872건, 임시 불량 0건 | 남은 미라벨 16857건
8회차 | 실제 학습 15 epoch | 임시 양품 1083건, 임시 불량 602건 | 남은 미라벨 15172건
9회차 | 실제 학습 15 epoch | 임시 양품 0건, 임시 불량 1517건 | 남은 미라벨 13655건
10회차 | 실제 학습 11 epoch | 임시 양품 0건, 임시 불량 1365건 | 남은 미라벨 12290건
11회차 | 실제 학습 11 epoch | 임시 양품 0건, 임시 불량 1229건 | 남은 미라벨 11061건
12회차 | 실제 학습 11 epoch | 임시 양품 0건, 임시 불량 1106건 | 남은 미라벨 9955건
13회차 | 실제 학습 11 epoch | 임시 양품 0건, 임시 불량 995건 | 남은 미라벨 8960건
14회차 | 실제 학습 21 epoch | 임시 양품 0건, 임시 불량 896건 | 남은 미라벨 8064건
15회차 | 실제 학습 14 epoch | 임시 양품 0건, 임시 불량 806건 | 남은 미라벨 7258건
16회차 | 실제 학

## 다른 알고리즘 적용, 결과 확인

In [24]:
import numpy as np
import pandas as pd
import tensorflow as tf

from catboost import CatBoostClassifier
from xgboost import XGBClassifier

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    fbeta_score,
    roc_auc_score,
    average_precision_score,
)

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam


def prepare_semi_data(product):
    X_train, y_train, X_test, y_test, X_unlabeled = (
        load_product_data(product)
    )

    X_fit, X_val, y_fit, y_val = train_test_split(
        X_train,
        y_train,
        test_size=0.2,
        stratify=y_train,
        random_state=42,
    )

    # 평가·검증 데이터와 완전히 동일한 입력값이
    # 임시 라벨 학습으로 들어오지 않도록 제외
    labeled_hash = set(
        pd.util.hash_pandas_object(
            pd.concat([X_train, X_test], ignore_index=True),
            index=False,
        )
    )

    unlabeled_hash = pd.util.hash_pandas_object(
        X_unlabeled,
        index=False,
    )

    X_unlabeled = (
        X_unlabeled.loc[
            ~unlabeled_hash.isin(labeled_hash)
        ]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    scaler = StandardScaler()

    X_fit = scaler.fit_transform(X_fit).astype("float32")
    X_val = scaler.transform(X_val).astype("float32")
    X_test = scaler.transform(X_test).astype("float32")
    X_unlabeled = scaler.transform(
        X_unlabeled
    ).astype("float32")

    y_fit = y_fit.to_numpy()
    y_val = y_val.to_numpy()
    y_test = y_test.to_numpy()

    n_pos = int(np.sum(y_fit == 1))
    n_neg = int(np.sum(y_fit == 0))

    if n_pos == 0:
        raise ValueError("학습 데이터에 불량 사례가 없습니다.")

    positive_weight = min(n_neg / n_pos, 20.0)

    print(
        f"{product.upper()} | "
        f"학습 불량 {n_pos}건, "
        f"검증 불량 {int(np.sum(y_val == 1))}건, "
        f"평가 불량 {int(np.sum(y_test == 1))}건, "
        f"미라벨 후보 {len(X_unlabeled)}건"
    )

    return (
        X_fit, y_fit,
        X_val, y_val,
        X_test, y_test,
        X_unlabeled,
        positive_weight,
    )


def select_pseudo_labels(
    X_unlabeled,
    probabilities,
    max_fraction=0.10,
    normal_cutoff=0.01,
    defect_cutoff=0.99,
):
    """
    전체 미라벨 후보의 최대 10%를 한 번만 선택.
    확률 0.01 이하 → 임시 양품
    확률 0.99 이상 → 임시 불량
    """
    probabilities = np.asarray(probabilities).ravel()

    candidates = pd.DataFrame({
        "row": np.arange(len(X_unlabeled)),
        "prob": probabilities,
    })

    candidates = candidates.loc[
        (candidates["prob"] <= normal_cutoff)
        | (candidates["prob"] >= defect_cutoff)
    ].copy()

    candidates["pseudo_label"] = (
        candidates["prob"] >= defect_cutoff
    ).astype(int)

    candidates["confidence"] = np.maximum(
        candidates["prob"],
        1 - candidates["prob"],
    )

    max_count = int(len(X_unlabeled) * max_fraction)

    selected = candidates.nlargest(
        max_count,
        "confidence",
    )

    selected_rows = selected["row"].to_numpy(dtype=int)

    X_pseudo = X_unlabeled[selected_rows]
    y_pseudo = selected["pseudo_label"].to_numpy(dtype=int)

    print(
        f"임시 라벨 채택 {len(y_pseudo)}건 | "
        f"양품 {int(np.sum(y_pseudo == 0))}건, "
        f"불량 {int(np.sum(y_pseudo == 1))}건"
    )

    return X_pseudo, y_pseudo


def choose_threshold_on_validation(model, X_val, y_val):
    val_prob = model.predict_proba(X_val)[:, 1]
    candidates = np.linspace(0.05, 0.95, 37)

    scores = []

    for threshold in candidates:
        pred = (val_prob >= threshold).astype(int)

        f2 = fbeta_score(
            y_val,
            pred,
            beta=2,
            zero_division=0,
        )

        fp = int(np.sum((y_val == 0) & (pred == 1)))
        fn = int(np.sum((y_val == 1) & (pred == 0)))

        scores.append((f2, -fp, threshold, fn))

    best = max(scores)

    print(
        f"검증 기준 {best[2]:.3f} | "
        f"검증 F2 {best[0]:.3f} | "
        f"검증 오탐 {-best[1]}건 | "
        f"검증 불량 누락 {best[3]}건"
    )

    return float(best[2])


def report_semi_result(name, model, X_val, y_val, X_test, y_test):
    print(f"\n[{name}]")

    threshold = choose_threshold_on_validation(
        model,
        X_val,
        y_val,
    )

    test_prob = model.predict_proba(X_test)[:, 1]
    test_pred = (test_prob >= threshold).astype(int)

    evaluate_model(y_test, test_pred)

    print(
        "확률 기반 ROC-AUC:",
        roc_auc_score(y_test, test_prob),
    )
    print(
        "Average Precision:",
        average_precision_score(y_test, test_prob),
    )

    return threshold

In [25]:
def run_catboost_to_xgboost(product="cn7"):
    (
        X_fit, y_fit,
        X_val, y_val,
        X_test, y_test,
        X_unlabeled,
        positive_weight,
    ) = prepare_semi_data(product)

    # 1. CatBoost가 임시 라벨 생성
    teacher = CatBoostClassifier(
        iterations=300,
        depth=4,
        learning_rate=0.03,
        loss_function="Logloss",
        scale_pos_weight=positive_weight,
        random_seed=42,
        verbose=False,
        allow_writing_files=False,
    )

    teacher.fit(X_fit, y_fit)

    unlabeled_prob = teacher.predict_proba(
        X_unlabeled
    )[:, 1]

    X_pseudo, y_pseudo = select_pseudo_labels(
        X_unlabeled,
        unlabeled_prob,
    )

    # 2. XGBoost가 최종 판정
    def make_xgboost():
        return XGBClassifier(
            n_estimators=100,
            max_depth=2,
            learning_rate=0.05,
            min_child_weight=3,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=positive_weight,
            random_state=42,
            n_jobs=-1,
        )

    baseline = make_xgboost()
    baseline.fit(X_fit, y_fit)

    semi_model = make_xgboost()

    if len(y_pseudo) > 0:
        X_combined = np.concatenate([X_fit, X_pseudo])
        y_combined = np.concatenate([y_fit, y_pseudo])

        # 공식 라벨 1.0, 임시 라벨 0.2
        weights = np.concatenate([
            np.ones(len(y_fit)),
            np.full(len(y_pseudo), 0.2),
        ])

        semi_model.fit(
            X_combined,
            y_combined,
            sample_weight=weights,
        )
    else:
        semi_model.fit(X_fit, y_fit)

    report_semi_result(
        f"{product.upper()} XGBoost: 공식 라벨만",
        baseline,
        X_val, y_val,
        X_test, y_test,
    )

    report_semi_result(
        f"{product.upper()} CatBoost 임시 라벨 → XGBoost",
        semi_model,
        X_val, y_val,
        X_test, y_test,
    )

    return teacher, baseline, semi_model


# 실행
cat_teacher, xgb_baseline, cat_to_xgb = (
    run_catboost_to_xgboost("cn7")
)

CN7: 라벨 학습 847건, 평가 364건, 미라벨 35239건
CN7 | 학습 불량 10건, 검증 불량 2건, 평가 불량 5건, 미라벨 후보 28771건
임시 라벨 채택 2877건 | 양품 2877건, 불량 0건

[CN7 XGBoost: 공식 라벨만]
검증 기준 0.200 | 검증 F2 0.500 | 검증 오탐 10건 | 검증 불량 누락 0건
혼동행렬:
 [[348  11]
 [  0   5]]
정확도: 0.9697802197802198
정밀도: 0.3125
재현율: 1.0
F1: 0.47619047619047616
ROC-AUC: 0.9846796657381616
확률 기반 ROC-AUC: 0.9866295264623954
Average Precision: 0.6095238095238096

[CN7 CatBoost 임시 라벨 → XGBoost]
검증 기준 0.175 | 검증 F2 0.476 | 검증 오탐 11건 | 검증 불량 누락 0건
혼동행렬:
 [[344  15]
 [  0   5]]
정확도: 0.9587912087912088
정밀도: 0.25
재현율: 1.0
F1: 0.4
ROC-AUC: 0.979108635097493
확률 기반 ROC-AUC: 0.9894150417827298
Average Precision: 0.6412698412698412


In [27]:
def run_keras_to_catboost(product="cn7"):
    (
        X_fit, y_fit,
        X_val, y_val,
        X_test, y_test,
        X_unlabeled,
        positive_weight,
    ) = prepare_semi_data(product)

    # --------------------------------------------------
    # 1. 임시 라벨을 붙일 작은 Keras 모델
    # --------------------------------------------------
    tf.keras.utils.set_random_seed(42)

    teacher = Sequential([
        Input(shape=(X_fit.shape[1],)),
        Dense(32, activation="relu"),
        Dropout(0.1),
        Dense(16, activation="relu"),
        Dense(1, activation="sigmoid"),
    ])

    teacher.compile(
        optimizer=Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
        ],
    )

    X_with_label = X_fit.copy()
    y_with_label = y_fit.copy()

    # Keras 교사용 가중치:
    # 실제 불량은 학습 데이터에서 적으므로 가중치를 높임
    teacher_weights = np.where(
        y_fit == 1,
        positive_weight,
        1.0,
    ).astype("float32")

    # CatBoost 최종 판정용 가중치:
    # 공식 라벨 1.0, 임시 라벨 0.2
    final_weights = np.ones(
        len(y_fit),
        dtype="float32",
    )

    without_label = X_unlabeled.copy()

    original_unlabeled_count = len(without_label)
    target_remaining = int(
        original_unlabeled_count * 0.10
    )

    round_number = 0
    total_pseudo_normal = 0
    total_pseudo_defect = 0

    # --------------------------------------------------
    # 2. 매 회차 남은 미라벨의 10%에 임시 라벨 부착
    #    처음 미라벨의 약 90%를 사용할 때 종료
    # --------------------------------------------------
    while len(without_label) > target_remaining:
        round_number += 1

        history = teacher.fit(
            X_with_label,
            y_with_label,
            sample_weight=teacher_weights,
            validation_data=(X_val, y_val),
            batch_size=32,
            epochs=100,
            callbacks=[
                EarlyStopping(
                    monitor="val_loss",
                    patience=10,
                    restore_best_weights=True,
                )
            ],
            verbose=0,
        )

        probabilities = teacher.predict(
            without_label,
            verbose=0,
        ).ravel()

        # 양품·불량 중 높은 예측 확률을 확신도로 사용
        confidence = np.maximum(
            probabilities,
            1 - probabilities,
        )

        # 매 회차 '남은 미라벨'의 10%
        count = max(
            1,
            int(len(without_label) * 0.10),
        )

        # 처음 미라벨의 약 90%까지만 사용
        count = min(
            count,
            len(without_label) - target_remaining,
        )

        selected = np.argsort(
            confidence
        )[::-1][:count]

        remaining_mask = np.ones(
            len(without_label),
            dtype=bool,
        )
        remaining_mask[selected] = False

        X_pseudo = without_label[selected]

        y_pseudo = (
            probabilities[selected] >= 0.5
        ).astype(int)

        # Keras 교사 모델의 다음 회차 학습에 사용
        pseudo_teacher_weights = (
            np.where(
                y_pseudo == 1,
                positive_weight,
                1.0,
            ) * 0.2
        ).astype("float32")

        # CatBoost 최종 모델에는 임시 라벨 신뢰도 0.2만 적용.
        # 불량 가중치는 CatBoost의 scale_pos_weight가 담당.
        pseudo_final_weights = np.full(
            len(y_pseudo),
            0.2,
            dtype="float32",
        )

        X_with_label = np.concatenate([
            X_with_label,
            X_pseudo,
        ])

        y_with_label = np.concatenate([
            y_with_label,
            y_pseudo,
        ])

        teacher_weights = np.concatenate([
            teacher_weights,
            pseudo_teacher_weights,
        ])

        final_weights = np.concatenate([
            final_weights,
            pseudo_final_weights,
        ])

        without_label = without_label[
            remaining_mask
        ]

        added_normal = int(
            np.sum(y_pseudo == 0)
        )
        added_defect = int(
            np.sum(y_pseudo == 1)
        )

        total_pseudo_normal += added_normal
        total_pseudo_defect += added_defect

        print(
            f"{round_number}회차 | "
            f"학습 "
            f"{len(history.history['loss'])} epoch | "
            f"임시 양품 {added_normal}건, "
            f"임시 불량 {added_defect}건 | "
            f"남은 미라벨 {len(without_label)}건"
        )

    print(
        f"\n총 {round_number}회차 | "
        f"임시 양품 {total_pseudo_normal}건, "
        f"임시 불량 {total_pseudo_defect}건"
    )

    # --------------------------------------------------
    # 3. 최종 판정용 CatBoost
    #    공식 라벨만 / 임시 라벨 추가 비교
    # --------------------------------------------------
    def make_catboost():
        return CatBoostClassifier(
            iterations=300,
            depth=4,
            learning_rate=0.03,
            loss_function="Logloss",
            scale_pos_weight=positive_weight,
            random_seed=42,
            verbose=False,
            allow_writing_files=False,
        )

    baseline = make_catboost()
    baseline.fit(X_fit, y_fit)

    semi_model = make_catboost()
    semi_model.fit(
        X_with_label,
        y_with_label,
        sample_weight=final_weights,
    )

    report_semi_result(
        f"{product.upper()} CatBoost: 공식 라벨만",
        baseline,
        X_val, y_val,
        X_test, y_test,
    )

    report_semi_result(
        f"{product.upper()} 작은 Keras 임시 라벨 → CatBoost",
        semi_model,
        X_val, y_val,
        X_test, y_test,
    )

    return teacher, baseline, semi_model


# CN7 실행
keras_teacher, cat_baseline, keras_to_cat = (
    run_keras_to_catboost("cn7")
)

# RG3 실행 시
# rg3_teacher, rg3_cat_baseline, rg3_keras_to_cat = (
#     run_keras_to_catboost("rg3")
# )

CN7: 라벨 학습 847건, 평가 364건, 미라벨 35239건
CN7 | 학습 불량 10건, 검증 불량 2건, 평가 불량 5건, 미라벨 후보 28771건
1회차 | 학습 37 epoch | 임시 양품 2877건, 임시 불량 0건 | 남은 미라벨 25894건
2회차 | 학습 41 epoch | 임시 양품 2589건, 임시 불량 0건 | 남은 미라벨 23305건
3회차 | 학습 11 epoch | 임시 양품 2330건, 임시 불량 0건 | 남은 미라벨 20975건
4회차 | 학습 12 epoch | 임시 양품 2097건, 임시 불량 0건 | 남은 미라벨 18878건
5회차 | 학습 12 epoch | 임시 양품 1887건, 임시 불량 0건 | 남은 미라벨 16991건
6회차 | 학습 15 epoch | 임시 양품 1699건, 임시 불량 0건 | 남은 미라벨 15292건
7회차 | 학습 14 epoch | 임시 양품 1529건, 임시 불량 0건 | 남은 미라벨 13763건
8회차 | 학습 20 epoch | 임시 양품 1376건, 임시 불량 0건 | 남은 미라벨 12387건
9회차 | 학습 11 epoch | 임시 양품 1238건, 임시 불량 0건 | 남은 미라벨 11149건
10회차 | 학습 12 epoch | 임시 양품 1114건, 임시 불량 0건 | 남은 미라벨 10035건
11회차 | 학습 14 epoch | 임시 양품 699건, 임시 불량 304건 | 남은 미라벨 9032건
12회차 | 학습 11 epoch | 임시 양품 0건, 임시 불량 903건 | 남은 미라벨 8129건
13회차 | 학습 13 epoch | 임시 양품 0건, 임시 불량 812건 | 남은 미라벨 7317건
14회차 | 학습 13 epoch | 임시 양품 0건, 임시 불량 731건 | 남은 미라벨 6586건
15회차 | 학습 20 epoch | 임시 양품 0건, 임시 불량 658건 | 남은 미라벨 5928건
16회차 | 학습 20 epoch | 임시 양품 0건, 임시 불량 592건 | 